# LunarLander Offline Reward Engineering Competition

 **Your task:** Write a **custom reward function** that ranks pre-recorded `LunarLander-v3` trajectories. Edit `custom_reward.py`, validate locally with the tools in this notebook, then upload a submission zip to a Google Form.

Local validation uses `validation_set.pkl` (12 trajectories with labels).

## Observation layout (`LunarLander-v3`)

| Index | Meaning |
|-------|---------|
| 0, 1 | x, y position |
| 2, 3 | x, y velocity |
| 4, 5 | angle, angular velocity |
| 6, 7 | left/right leg ground contact (0 or 1) |

## Action space

| Action | Meaning |
|--------|---------|
| 0 | do nothing (coast) |
| 1 | fire left orientation engine |
| 2 | fire main (bottom) engine |
| 3 | fire right orientation engine |

## 1. Team name and upload

1. Set your **team name** below (used as the folder name in your submission zip).
2. Run the cell and upload **`participant_template.zip`** when prompted.

In [ ]:
import re

TEAM_NAME = "YourTeamName"  # <-- edit this


def sanitize_team_name(name: str) -> str:
    cleaned = name.strip().replace(" ", "_")
    if not cleaned:
        raise ValueError("TEAM_NAME must not be empty.")
    if re.search(r'[/\\<>:"|?*]', cleaned):
        raise ValueError(f"TEAM_NAME contains invalid path characters: {cleaned!r}")
    return cleaned


SANITIZED_TEAM_NAME = sanitize_team_name(TEAM_NAME)
print(f"Submission folder name: {SANITIZED_TEAM_NAME}")

from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("No file uploaded. Upload participant_template.zip.")

## 2. Unzip and install dependencies

In [ ]:
import os
import zipfile
from pathlib import Path

zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise FileNotFoundError("Upload participant_template.zip (a .zip file).")

zip_path = Path(zip_names[0])
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(".")

if Path("participant_template/custom_reward.py").exists():
    work_dir = Path("participant_template")
elif Path("custom_reward.py").exists():
    work_dir = Path(".")
else:
    raise FileNotFoundError(
        "Unzip did not produce custom_reward.py. "
        "Ensure the zip contains the participant_template folder."
    )

os.chdir(work_dir)
print(f"Working directory: {Path.cwd().resolve()}")
print("Files:", sorted(path.name for path in Path(".").iterdir() if path.is_file()))

In [ ]:
!pip install -q -r requirements.txt

## 3. Implement your reward

Edit the cell below and **re-run it** whenever you change your reward. All validation tools load `custom_reward.py` from disk.

The starter returns `0.0` everywhere — replace it with your reward engineering.

In [ ]:
%%writefile custom_reward.py
"""Participant submission: custom reward function for offline trajectory ranking."""

from __future__ import annotations

import numpy as np


def custom_reward(
    state: np.ndarray,
    action: int,
    next_state: np.ndarray,
    terminated: bool,
    truncated: bool,
) -> float:
    """Compute shaped reward for one LunarLander-v3 transition.

    Parameters
    ----------
    state : np.ndarray, shape (8,)
        Pre-step observation.
    action : int
        Discrete action taken (0..3).
    next_state : np.ndarray, shape (8,)
        Post-step observation.
    terminated : bool
        True if episode ended due to landing/crash.
    truncated : bool
        True if episode ended due to time limit.

    Returns
    -------
    float
        Scalar reward for this transition (summed over steps = predicted return).

    Observation layout (LunarLander-v3)
    -----------------------------------
    [0] x position of lander center (horizontal offset from pad; 0 = centered)
    [1] y position / altitude (higher value = higher above ground)
    [2] x velocity (m/s)
    [3] y velocity (m/s; negative = falling)
    [4] angle in radians (0 = upright)
    [5] angular velocity (rad/s; spin rate)
    [6] left leg ground contact (1.0 if touching ground)
    [7] right leg ground contact (1.0 if touching ground)

    Action space (Discrete)
    -----------------------
    0 = do nothing (coast)
    1 = fire left orientation engine
    2 = fire main (bottom) engine
    3 = fire right orientation engine
    """
    # TODO: Replace this starter with your reward engineering.
    # Ideas:
    #   - Reward progress toward pad center (state[0]) and descent (state[1])
    #   - Penalize tilt (state[4]) and angular velocity (state[5])
    #   - Penalize engine use (actions 1, 2, 3) for fuel efficiency
    #   - Add terminal bonus for safe landing, penalty for crash
    _ = (state, action, next_state, terminated, truncated)
    return 0.0

## 4. Sanity tests

These checks assert basic reward behavior (landing beats crash, coasting beats main-engine hover).

The starter template **will fail** until you implement a real reward. Re-run after editing Section 3.

In [ ]:
!python test_sanity.py

## 5. Local validation grader

Scores your reward on `validation_set.pkl` only (12 trajectories, 3 per policy tier). Prints Spearman ρ and a per-trajectory table.

Iterate on your reward until ρ improves. This does **not** use the hidden evaluation set.

In [ ]:
!python local_grader.py

## 6. Visual debugger — console mode

Replays an **exact offline validation trajectory** step by step with an ASCII altitude bar and per-step rewards.

Validation indices **0–11** (3 trajectories per tier). Try tier 1 (strong) vs tier 4 (weak) trajectories.

In [ ]:
TRAJECTORY_INDEX = 0

!python visual_debugger.py --mode console --trajectory-index {TRAJECTORY_INDEX}

## 7. Visual debugger — video mode

Runs a **new live rollout** in Box2D (not an exact replay of offline data). The MP4 embeds automatically in Colab when you run the cell below (in-process Python, not `!python`).

Use console mode above to inspect exact offline trajectories.

In [ ]:
from visual_debugger import run_video

# Run in the notebook kernel so Colab can embed the MP4 (subprocess !python cannot).
run_video(seed=0, policy="random", output="debug_episode.mp4")

## 8. Download submission

Packages `{team_name}/custom_reward.py` into a zip for the organizer. The folder name comes from your **TEAM_NAME** in Section 1.

Re-run Section 1 first if you changed your team name.

In [ ]:
import importlib.util
import shutil
import subprocess
from pathlib import Path

from google.colab import files

reward_path = Path("custom_reward.py")
if not reward_path.exists():
    raise FileNotFoundError("custom_reward.py not found. Run the writefile cell in Section 3.")

spec = importlib.util.spec_from_file_location("submission_check", reward_path)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)
if not hasattr(module, "custom_reward"):
    raise AttributeError("custom_reward.py must define custom_reward(...)")

sanity = subprocess.run(["python", "test_sanity.py"], capture_output=True, text=True)
if sanity.returncode != 0:
    print("WARNING: sanity tests did not pass. Fix your reward before submitting.")
    if sanity.stdout:
        print(sanity.stdout)
    if sanity.stderr:
        print(sanity.stderr)
else:
    print("Sanity tests passed.")

team_dir = Path(SANITIZED_TEAM_NAME)
if team_dir.exists():
    shutil.rmtree(team_dir)
team_dir.mkdir()
shutil.copy2(reward_path, team_dir / "custom_reward.py")

archive_base = f"{SANITIZED_TEAM_NAME}_submission"
archive_path = Path(f"{archive_base}.zip")
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(archive_base, "zip", root_dir=".", base_dir=SANITIZED_TEAM_NAME)

files.download(str(archive_path))
print(f"Downloaded {archive_path.name}")
print(f"Zip layout: {SANITIZED_TEAM_NAME}/custom_reward.py")

## Submission checklist

- [ ] **Team name** in Section 1 matches the folder inside your zip.
- [ ] **Sanity tests** pass (Section 4).
- [ ] Submit the downloaded zip (or only `custom_reward.py` inside the named team folder).
- [ ] Do **not** submit `validation_set.pkl`, helper scripts, or this notebook.
- [ ] Submit the downloaded zip file here: https://forms.gle/qYvecf23ottSedVf8